In [33]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.preprocessing import MinMaxScaler

In [34]:
BASE_DIR = Path.cwd().parent

DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_EXTERNAL = BASE_DIR / "data" / "external"

In [35]:
df = pd.read_csv(
    DATA_PROCESSED / "dataset_maestro_granada_v2.csv"
)

print(df.shape)

df.head()

(174, 24)


,codigo_ine,municipio,poblacion_2025,n_poligonos_residenciales,superficie_residencial_m2,superficie_residencial_ha,superficie_media_poligono_m2,superficie_mediana_poligono_m2,superficie_max_poligono_m2,perimetro_total_m,...,area_m2_desarrollo,sunc_m2_por_hab,urbanizable_m2_por_hab,comarca,precio_venta_eur_m2,precio_alquiler_eur_m2_mes,poblacion_aprox,tipo_dato,fuente,fecha
0,18905,"Gabias, Las",23584,1.0,3.682035e+06,368.203480,3.682035e+06,3.682035e+06,3.682035e+06,40983.790385,...,0.000000e+00,NaN,NaN,Área Metropolitana,1601.0,8.40,25600.0,V,Idealista/Indomio/Fotocasa 2026,2026-05
1,18089,Guadix,18881,7.0,3.460928e+06,346.092770,4.944182e+05,1.185983e+05,2.848796e+06,43493.539742,...,7.912660e+05,23.199142,18.708914,Comarca de Guadix,839.0,NaN,18800.0,V,Idealista/Indomio/Fotocasa 2026,2026-05
2,18022,Atarfe,20914,1.0,3.364040e+06,336.403955,3.364040e+06,3.364040e+06,3.364040e+06,37294.919233,...,3.669618e+06,1.602372,173.859902,Vega de Granada,1380.0,NaN,17300.0,V,Idealista/Indomio/Fotocasa 2026,2026-05
3,18029,Benamaurel,2235,9.0,3.295178e+06,329.517829,3.661309e+05,1.089552e+05,1.376323e+06,32558.922141,...,4.764880e+05,13.460850,199.732886,Comarca de Baza,510.0,NaN,1800.0,E,Estimación por comarca/tipología,2026-05
4,18003,Albolote,19768,1.0,2.949836e+06,294.983553,2.949836e+06,2.949836e+06,2.949836e+06,42092.590074,...,2.876213e+06,41.905909,103.592530,Área Metropolitana,1622.0,8.88,19800.0,V,Idealista/Indomio/Fotocasa 2026,2026-05


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174 entries, 0 to 173
Data columns (total 24 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   codigo_ine                      174 non-null    int64  
 1   municipio                       174 non-null    object 
 2   poblacion_2025                  174 non-null    int64  
 3   n_poligonos_residenciales       128 non-null    float64
 4   superficie_residencial_m2       128 non-null    float64
 5   superficie_residencial_ha       128 non-null    float64
 6   superficie_media_poligono_m2    128 non-null    float64
 7   superficie_mediana_poligono_m2  128 non-null    float64
 8   superficie_max_poligono_m2      128 non-null    float64
 9   perimetro_total_m               128 non-null    float64
 10  suelo_residencial_m2_por_hab    128 non-null    float64
 11  municipio_key                   174 non-null    object 
 12  area_m2_SUNC                    90 n

In [37]:
df["area_m2_desarrollo"] = (
    df["area_m2_SUNC"].fillna(0)
    + df["area_m2_Urbanizable"].fillna(0)
)

In [38]:
df["reserva_suelo_hab"] = np.where(
    df["poblacion_2025"] > 0,
    df["area_m2_desarrollo"] / df["poblacion_2025"],
    np.nan
)

In [39]:
df["indice_expansion"] = np.where(
    df["superficie_residencial_m2"] > 0,
    df["area_m2_desarrollo"] /
    df["superficie_residencial_m2"],
    np.nan
)

In [40]:
df["densidad_residencial"] = np.where(
    df["superficie_residencial_ha"] > 0,
    df["poblacion_2025"] /
    df["superficie_residencial_ha"],
    np.nan
)

In [41]:
df[
    [
        "municipio",
        "poblacion_2025",
        "area_m2_desarrollo",
        "reserva_suelo_hab",
        "indice_expansion",
        "densidad_residencial"
    ]
].head()

,municipio,poblacion_2025,area_m2_desarrollo,reserva_suelo_hab,indice_expansion,densidad_residencial
0,"Gabias, Las",23584,0.000000e+00,0.000000,0.000000,64.051540
1,Guadix,18881,7.912660e+05,41.908056,0.228628,54.554737
2,Atarfe,20914,3.669618e+06,175.462274,1.090837,62.169305
3,Benamaurel,2235,4.764880e+05,213.193736,0.144602,6.782638
4,Albolote,19768,2.876213e+06,145.498439,0.975042,67.013906


In [42]:
df.to_csv(
    DATA_PROCESSED / "dataset_feature_engineering_v2.csv",
    index=False
)

print("OK")

OK


In [43]:
variables_score = [
    "poblacion_2025",
    "precio_venta_eur_m2",
    "precio_alquiler_eur_m2_mes",
    "suelo_residencial_m2_por_hab",
    "reserva_suelo_hab",
    "indice_expansion",
    "densidad_residencial"
]

In [44]:
df_score = df.copy()

for col in variables_score:
    df_score[col] = df_score[col].fillna(
        df_score[col].median()
    )

In [45]:
scaler = MinMaxScaler()

df_score[
    [f"{c}_norm" for c in variables_score]
] = scaler.fit_transform(
    df_score[variables_score]
)

In [46]:
df_score["precio_venta_score"] = (
    1 - df_score["precio_venta_eur_m2_norm"]
)

df_score["precio_alquiler_score"] = (
    1 - df_score["precio_alquiler_eur_m2_mes_norm"]
)

In [52]:
df_score["coopscore"] = (

    df_score["poblacion_2025_norm"] * 0.30 +

    df_score["suelo_residencial_m2_por_hab_norm"] * 0.10 +

    df_score["reserva_suelo_hab_norm"] * 0.15 +

    df_score["indice_expansion_norm"] * 0.10 +

    df_score["densidad_residencial_norm"] * 0.10 +

    df_score["precio_venta_score"] * 0.10 +

    df_score["precio_alquiler_score"] * 0.15

) * 100

In [53]:
df_score["factor_escala"] = np.log1p(
    df_score["poblacion_2025"]
)

df_score["coopscore"] *= (
    df_score["factor_escala"] /
    df_score["factor_escala"].max()
)

In [54]:
ranking = (
    df_score
    .sort_values(
        "coopscore",
        ascending=False
    )
)

ranking[
    [
        "municipio",
        "coopscore",
        "poblacion_2025"
    ]
].head(20)

,municipio,coopscore,poblacion_2025
141,Granada,46.587855,233975
37,Motril,27.714906,59862
68,Pinos Genil,24.823665,1642
21,Alfacar,18.105684,5834
3,Benamaurel,17.420042,2235
124,Torrenueva Costa,16.649315,3207
8,Baza,16.612392,20587
5,Loja,16.497473,20951
59,Lanjarón,16.259876,3708
2,Atarfe,15.518991,20914


In [55]:
ranking["coopscore"].describe()

count    174.000000
mean      10.900246
std        3.880816
min        6.630383
25%        9.132266
50%       10.136383
75%       11.710695
max       46.587855
Name: coopscore, dtype: float64

In [56]:
ranking.to_csv(
    DATA_PROCESSED / "dataset_modelado_v3.csv",
    index=False
)

print("Dataset modelado generado")

Dataset modelado generado


In [57]:
df_score[
    df_score["municipio"] == "Salobreña"
][[
    "municipio",
    "coopscore",
    "poblacion_2025",
    "precio_venta_eur_m2",
    "precio_alquiler_eur_m2_mes",
    "suelo_residencial_m2_por_hab",
    "reserva_suelo_hab",
    "indice_expansion",
    "densidad_residencial"
]]

,municipio,coopscore,poblacion_2025,precio_venta_eur_m2,precio_alquiler_eur_m2_mes,suelo_residencial_m2_por_hab,reserva_suelo_hab,indice_expansion,densidad_residencial
6,Salobreña,9.257491,12760,2173.0,9.1,199.828073,0.0,0.0,50.043019


In [58]:
df_score[
    df_score["municipio"] == "Salobreña"
][[
    "poblacion_2025_norm",
    "precio_venta_score",
    "precio_alquiler_score",
    "suelo_residencial_m2_por_hab_norm",
    "reserva_suelo_hab_norm",
    "indice_expansion_norm",
    "densidad_residencial_norm",
    "coopscore"
]]

,poblacion_2025_norm,precio_venta_score,precio_alquiler_score,suelo_residencial_m2_por_hab_norm,reserva_suelo_hab_norm,indice_expansion_norm,densidad_residencial_norm,coopscore
6,0.053974,0.342923,0.351753,0.128862,0.0,0.0,0.049245,9.257491


In [59]:
# ============================================================
# 19. CALIDAD DEL DATO URBANÍSTICO
# ============================================================

df["tiene_suelo_residencial"] = df["superficie_residencial_m2"].notna().astype(int)

df["tiene_sunc_urbanizable"] = (
    df["area_m2_SUNC"].notna() | 
    df["area_m2_Urbanizable"].notna()
).astype(int)

df["calidad_dato_suelo"] = (
    df["tiene_suelo_residencial"] +
    df["tiene_sunc_urbanizable"]
)

df[[
    "municipio",
    "tiene_suelo_residencial",
    "tiene_sunc_urbanizable",
    "calidad_dato_suelo"
]].head()

,municipio,tiene_suelo_residencial,tiene_sunc_urbanizable,calidad_dato_suelo
0,"Gabias, Las",1,0,1
1,Guadix,1,1,2
2,Atarfe,1,1,2
3,Benamaurel,1,1,2
4,Albolote,1,1,2


In [61]:
for f in DATA_EXTERNAL.iterdir():
    print(f.name)

distancia_pueblos.csv
precios_vivienda_granada.csv
suelo_granada.csv
suelo_granada_clasificado.csv
suelo_granada_residencial.csv


In [64]:
distancias = pd.read_csv(
    DATA_EXTERNAL / "distancia_pueblos.csv",
    sep = ";"
)

print(distancias.shape)
print(distancias.columns.tolist())
distancias.head()

(168, 3)
['localidad', 'Distancia en Kms Capital', 'Distancia en kms a la costa']


,localidad,Distancia en Kms Capital,Distancia en kms a la costa
0,Agron,36.41,67.9
1,Alamedilla,72.89,106.9
2,Albolote,9.78,71.1
3,Albondon,114.59,43.9
4,Albunan,63.81,41.6


In [65]:
distancias.columns = [
    "municipio",
    "distancia_capital_km",
    "distancia_costa_km"
]

distancias.head()

,municipio,distancia_capital_km,distancia_costa_km
0,Agron,36.41,67.9
1,Alamedilla,72.89,106.9
2,Albolote,9.78,71.1
3,Albondon,114.59,43.9
4,Albunan,63.81,41.6


In [68]:
import unicodedata

def normalizar_municipio(x):

    if pd.isna(x):
        return x

    x = str(x).upper().strip()

    x = ''.join(
        c for c in unicodedata.normalize('NFD', x)
        if unicodedata.category(c) != 'Mn'
    )

    equivalencias = {
        "GABIAS, LAS": "LAS GABIAS",
        "MALAHA, LA": "LA MALAHA",
        "VALLE, EL": "EL VALLE",
        "PEZA, LA": "LA PEZA",
        "PUEBLA DE DON FADRIQUE": "PUEBLA DE D. FADRIQUE"
    }

    return equivalencias.get(x, x)

In [69]:
distancias["municipio_key"] = (
    distancias["municipio"]
    .apply(normalizar_municipio)
)

df_score["municipio_key"] = (
    df_score["municipio"]
    .apply(normalizar_municipio)
)

In [70]:
df_score = df_score.merge(
    distancias[
        [
            "municipio_key",
            "distancia_capital_km",
            "distancia_costa_km"
        ]
    ],
    on="municipio_key",
    how="left"
)

In [71]:
df_score[
    [
        "distancia_capital_km",
        "distancia_costa_km"
    ]
].isna().sum()

distancia_capital_km    17
distancia_costa_km      18
dtype: int64

In [72]:
sin_distancia = df_score[
    df_score["distancia_capital_km"].isna()
][["municipio"]]

sin_distancia

,municipio
0,"Gabias, Las"
43,Valderrubio
62,"Malahá, La"
63,"Calahorra, La"
71,Torre-Cardela
81,"Valle, El"
85,"Peza, La"
93,Domingo Pérez de Granada
95,Dehesas Viejas
106,Játar


In [73]:
df_score["distancia_capital_km"] = (
    df_score["distancia_capital_km"]
    .fillna(df_score["distancia_capital_km"].median())
)

df_score["distancia_costa_km"] = (
    df_score["distancia_costa_km"]
    .fillna(df_score["distancia_costa_km"].median())
)

In [75]:
from sklearn.preprocessing import MinMaxScaler

def minmax_score(df, col, invert=False):
    serie = df[[col]].copy()
    serie[col] = serie[col].fillna(serie[col].median())

    scaler = MinMaxScaler()
    valores = scaler.fit_transform(serie)[..., 0]

    if invert:
        valores = 1 - valores

    return valores

In [76]:
df_score = df.copy()

df_score["score_poblacion"] = minmax_score(df_score, "poblacion_2025")
df_score["score_suelo_residencial"] = minmax_score(df_score, "suelo_residencial_m2_por_hab")
df_score["score_reserva_suelo"] = minmax_score(df_score, "reserva_suelo_hab")
df_score["score_expansion"] = minmax_score(df_score, "indice_expansion")
df_score["score_densidad"] = minmax_score(df_score, "densidad_residencial")

df_score["score_precio_venta"] = minmax_score(
    df_score,
    "precio_venta_eur_m2",
    invert=True
)

df_score["score_precio_alquiler"] = minmax_score(
    df_score,
    "precio_alquiler_eur_m2_mes",
    invert=False
)

df_score["score_calidad_suelo"] = minmax_score(
    df_score,
    "calidad_dato_suelo"
)

In [77]:
# ============================================================
# 23. ÍNDICES PARCIALES
# ============================================================

df_score["IVU"] = (
    df_score["score_suelo_residencial"] * 0.20 +
    df_score["score_reserva_suelo"] * 0.25 +
    df_score["score_expansion"] * 0.25 +
    df_score["score_densidad"] * 0.10 +
    df_score["score_calidad_suelo"] * 0.20
) * 100

df_score["IVEC"] = (
    df_score["score_poblacion"] * 0.45 +
    df_score["score_precio_venta"] * 0.20 +
    df_score["score_precio_alquiler"] * 0.25 +
    df_score["score_densidad"] * 0.10
) * 100

df_score[["municipio", "IVU", "IVEC"]].head()

,municipio,IVU,IVEC
0,"Gabias, Las",12.631715,30.756813
1,Guadix,24.023057,36.227069
2,Atarfe,27.732469,32.554620
3,Benamaurel,43.183368,35.003836
4,Albolote,26.941776,31.170308


In [78]:
ranking = (
    df_score
    .sort_values(
        "CoopScore_3",
        ascending=False
    )
)

ranking[
    [
        "municipio",
        "IVU",
        "IVEC",
        "IAL",
        "CoopScore_3"
    ]
].head(25)

KeyError: 'CoopScore_3'